# 03 · Staging: limpeza, tipagem e deduplicação

O notebook 01 faz o diagnóstico e aqui aplica o tratamento.
lógica em `src/etl/cleaning.py`
funçoes testadas `tests/unit/test_cleaning.py` e `src/etl/staging_transform.py`

Tratamentos principais:
| Objetivo | O que faz |
|---|---|
| **Strings** | acento/caixa padronizados em cidade, comentários livres só perdem quebras de linha e espaço extra |
| **CEP** | volta a ser texto de 5 dígitos, UF que não bate com a faixa do CEP é sinalizada |
| **Categorias** | traduzidas para inglês e agrupadas em macro-categorias, sem categoria vira `unknown` |
| **Tipagem** | datas em texto -> `datetime`, IDs -> `string`, flags -> `boolean` |
| **Números** | peso zero e outros valores impossiveis viram `NULL` e são imputados pela mediana da categoria |
| **Nulos** | mantem por motivo de negócio ou viram uma flag de qualidade |

> Cada decisão fica registrada em `staging.dq_issues` lida em qualidade dos dados no dashboard

In [1]:
# Configuracao inicial

import os
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root():
    current = Path.cwd().resolve()
    for p in [current] + list(current.parents):
        if (p / "src").exists() and (p / "data").exists() and (p / "config").exists():
            return p
    raise RuntimeError("Raiz do projeto nao encontrada")


def project_path(*segments):
    return find_project_root().joinpath(*segments)


root = find_project_root()
os.chdir(root)
print(f"Diretorio de trabalho: {root}")


Diretorio de trabalho: C:\Users\user\Downloads\Códigos\olist-ecommerce-pipeline\Template


## Um exemplo rápido

In [2]:
from src.etl.cleaning import standardize_zip, standardize_city, translate_category, coerce_numeric

translation_exemplo = pd.DataFrame({"product_category_name": ["eletronicos"], "product_category_name_english": ["electronics"]})

exemplos = pd.DataFrame({
    "antes": ["1046", "sao paulo/sp", "eletronicos", 0.0, "não é número"],
    "depois": [
        standardize_zip(pd.Series(["1046"])).iloc[0],
        standardize_city(pd.Series(["sao paulo/sp"]), pd.Series(["SP"])).iloc[0],
        translate_category(pd.Series(["eletronicos"]), translation_exemplo).iloc[0],
        coerce_numeric(pd.Series([0.0]), minimum=0.01).iloc[0],
        coerce_numeric(pd.Series(["não é número"])).iloc[0],
    ],
    "frente": ["CEP (texto de 5 dígitos)", "Cidade (padronizada)", "Categoria (traduzida)",
               "Número (0 é impossível, vira NULL p/ imputar)", "Número (texto inválido vira NULL)"],
})
exemplos

,antes,depois,frente
0,1046,01046,CEP (texto de 5 dígitos)
1,sao paulo/sp,sao paulo,Cidade (padronizada)
2,eletronicos,electronics,Categoria (traduzida)
3,0.0,NaN,"Número (0 é impossível, vira NULL p/ imputar)"
4,não é número,<NA>,Número (texto inválido vira NULL)


## Um exemplo completo: geolocation

Agora com uma tabela real inteira, não só um valor isolado. `raw.geolocation` tem ~1 milhão de linhas
coordenadas de GPS por CEP, incluindo duplicatas e pontos fora do Brasil. Limpeza agrega tudo
para uma linha por CEP

### Antes

In [6]:
from src.etl.db import get_engine
from src.etl.staging_transform import StagingTransformer

engine = get_engine()
raw_geo = pd.read_sql("SELECT * FROM raw.geolocation", engine)
print("Linhas antes:", len(raw_geo))
raw_geo.head()

Linhas antes: 1000163


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


### Depois

`clean_geolocation` é uma função `DataFrame -> DataFrame` sem acesso a banco

In [7]:
clean_geo = StagingTransformer.clean_geolocation(raw_geo)
print("Linhas depois:", len(clean_geo))
clean_geo.head()

Linhas depois: 19015


,geolocation_zip_code_prefix,geolocation_city,geolocation_state,avg_lat,avg_lng,n_points,n_points_discarded,geolocation_region,zip_state_mismatch
0,01001,sao paulo,SP,-23.550190,-46.634024,26,0,Sudeste,False
1,01002,sao paulo,SP,-23.548146,-46.634979,13,0,Sudeste,False
2,01003,sao paulo,SP,-23.548994,-46.635731,17,0,Sudeste,False
3,01004,sao paulo,SP,-23.549799,-46.634757,22,0,Sudeste,False
4,01005,sao paulo,SP,-23.549456,-46.636733,25,0,Sudeste,False


## Construindo toda a camada staging

`StagingTransformer.build_all()` 
na ordem certa: geolocation primeiro, pois `customers`/`sellers` usam ela para reparar cidade a partir do CEP

In [8]:
transformer = StagingTransformer(engine)
row_counts = transformer.build_all()
row_counts

{'stg_geolocation': 19015,
 'stg_customers': 99441,
 'stg_sellers': 3095,
 'stg_orders': 99441,
 'stg_order_items': 112650,
 'stg_order_payments': 103886,
 'stg_order_reviews': 99224,
 'stg_products': 32951}

## O que foi registrado

Cada `dq.add(...)` chamado durante a limpeza virou uma linha em `staging.dq_issues`.
Um resumo por ação (corrigido, imputado, anulado, sinalizado...)

In [9]:
pd.read_sql(
    "SELECT action, COUNT(*) AS regras, SUM(rows_affected) AS linhas "
    "FROM staging.dq_issues WHERE rows_affected > 0 GROUP BY action ORDER BY linhas DESC",
    engine,
)

,action,regras,linhas
0,dropped,4,1243760.0
1,fixed,16,358500.0
2,kept,5,166428.0
3,flagged,20,14908.0
4,imputed,6,629.0
5,nullified,6,48.0
